# Database migration notebook
Notebook för ETL-migrering med städning av databas.
Uppdatera ny version av SQLite databas fil och ersätta nuvarande SQLite databasfil med nyare version
Uppdatera music_warehouse.duckdb med nya rena databasen
----
Kolla först vilka tables i nya databasen som finns
Gör de ändringarna som behövs
Uppdatera men behåll gamla filen som backup. Namnger databasfilerna med  

`powerbi_warehouse_OLD.db` 
`music_warehouse_OLD.duckdb`

In [2]:
import duckdb
import os

In [6]:
# 1. Starta en tillfällig DuckDB i minnet
con = duckdb.connect()
con.execute("INSTALL sqlite;")
con.execute("LOAD sqlite;")

# 2. Koppla upp mot din SQLite-fil
sqlite_path = "../data/powerbi_warehouse.db"
con.execute(f"ATTACH '{sqlite_path}' AS sqlite_db (TYPE SQLITE);")

# 3. Använd DuckDB:s standardiserade 'information_schema' för att hitta tabellerna
query = """
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_catalog = 'sqlite_db';
"""
tables_df = con.execute(query).df()

print("Här är de exakta tabellnamnen inuti din SQLite-fil:\n")
print(tables_df)

con.close()

Här är de exakta tabellnamnen inuti din SQLite-fil:

                  table_name
0              spotify_daily
1          historical_charts
2                    top_200
3              dim_geography
4            top_200_cleaned
5      dim_geography_cleaned
6  historical_charts_cleaned


In [9]:
def migrate_and_clean():
    print("Startar städning och migrering tillbaka till DuckDB...")
    
    # 1. Skapa din helt nya, rena DuckDB-fil
    new_duckdb_path = "../data/music_warehouse.duckdb"
    
    if os.path.exists(new_duckdb_path):
        os.remove(new_duckdb_path)
        
    con = duckdb.connect(new_duckdb_path)
    
    con.execute("INSTALL sqlite;")
    con.execute("LOAD sqlite;")
    
    # 2. Koppla upp mot din SQLite-fil
    sqlite_path = "../data/powerbi_warehouse.db"
    con.execute(f"ATTACH '{sqlite_path}' AS sqlite_db (TYPE SQLITE);")
    
    print("Kopierar in de RENA tabellerna och döper om dem till standardnamn...")
    
    try:
        # 3. Mappningen: Hämta från SQLite -> Skapa i DuckDB
        
        print(" -> Skapar silver_spotify_daily (oförändrad)...")
        con.execute("""
            CREATE TABLE silver_spotify_daily AS 
            SELECT * FROM sqlite_db.spotify_daily;
        """)
        
        print(" -> Skapar silver_historical_charts (från cleaned)...")
        con.execute("""
            CREATE TABLE silver_historical_charts AS 
            SELECT * FROM sqlite_db.historical_charts_cleaned;
        """)
        
        print(" -> Skapar silver_top_200_historical (från cleaned)...")
        con.execute("""
            CREATE TABLE silver_top_200_historical AS 
            SELECT * FROM sqlite_db.top_200_cleaned;
        """)
        
        print(" -> Skapar dim_geography (från cleaned)...")
        con.execute("""
            CREATE TABLE dim_geography AS 
            SELECT * FROM sqlite_db.dim_geography_cleaned;
        """)
        
        print("\nMIGRERING KLAR! Nya rena databasen heter music_warehouse.duckdb")
        
    except Exception as e:
        print(f"\nEtt fel uppstod: {e}")
    finally:
        con.close()

if __name__ == "__main__":
    migrate_and_clean()

Startar städning och migrering tillbaka till DuckDB...
Kopierar in de RENA tabellerna och döper om dem till standardnamn...
 -> Skapar silver_spotify_daily (oförändrad)...
 -> Skapar silver_historical_charts (från cleaned)...
 -> Skapar silver_top_200_historical (från cleaned)...
 -> Skapar dim_geography (från cleaned)...

MIGRERING KLAR! Nya rena databasen heter music_warehouse.duckdb
